# NeMo diarization and Faster Whisper ASR

Run top to bottom in a **fresh local Jupyter kernel** with the matching route environment from README. Put one extracted audio/video file path in ignored `local_config.json` or set `TRAUMA_INPUT_FILE`. `DENOISE=True` applies the selected `DENOISE_METHOD`; `False` copies the normalized WAV unchanged. Both versions and the input hash are saved under `runs/`. Section 2 provides short inline audio players and a full-recording HTML comparison.

Outputs are draft anonymous speaker labels, not verified identities or clinical roles. Check the audio before interpreting them. No input recordings or outputs belong in Git.


## 1. Settings and GPU check

In [ ]:
ROUTE = "nemo_fw"
from pathlib import Path
import sys
import json
import os
from pipeline_common import (prepare_audio, gpu_preflight, save_turns, save_summary,
                             write_audio_preview, make_inline_audio_clips)

# Prefer an environment variable or ignored local_config.json to keep private paths out of Git.
INPUT_FILE = os.environ.get("TRAUMA_INPUT_FILE", "")
if not INPUT_FILE and Path("local_config.json").is_file():
    INPUT_FILE = json.loads(Path("local_config.json").read_text(encoding="utf-8")).get("input_file", "")
OUTPUT_ROOT = "runs"
AUDIO_STREAM = 0
DENOISE = False  # True / False
DENOISE_METHOD = "noisereduce"  # "noisereduce" or "afftdn"
DENOISE_STRENGTH = 0.8  # noisereduce: 0.0..1.0
NOISE_REDUCTION_DB = 12  # FFmpeg afftdn nr; 1..30; only used if DENOISE=True
PREVIEW_START_SECONDS = 0  # choose where to listen in the recording
PREVIEW_DURATION_SECONDS = 20  # 1..60; bounded notebook audio output

if not INPUT_FILE:
    raise ValueError("Set TRAUMA_INPUT_FILE or create ignored local_config.json with input_file")
print("Notebook Python:", sys.executable)
print("Route:", ROUTE, "| denoise:", DENOISE, "| method:", DENOISE_METHOD)


## 2. Convert video/audio to WAV; optionally denoise

In [ ]:
run_dir, model_wav, manifest = prepare_audio(
    INPUT_FILE, OUTPUT_ROOT, ROUTE, denoise=DENOISE,
    audio_stream=AUDIO_STREAM, noise_reduction_db=NOISE_REDUCTION_DB,
    denoise_method=DENOISE_METHOD, denoise_strength=DENOISE_STRENGTH,
)
duration_seconds = manifest["duration_seconds"]
print(f"Model input: {model_wav} ({duration_seconds:.2f}s)")
print(f"Denoising: {manifest['denoise_filter'] or 'off'}")
print(f"Output: {run_dir}")
preview_path = write_audio_preview(run_dir, denoise=DENOISE, method=DENOISE_METHOD)
print(f"Open this file in a browser to compare both audio players: {preview_path}")
preview_original, preview_model = make_inline_audio_clips(
    run_dir, start_seconds=PREVIEW_START_SECONDS,
    duration_seconds=PREVIEW_DURATION_SECONDS,
)
from IPython.display import Audio, Markdown, display
display(Markdown(f"**Original audio** — {PREVIEW_START_SECONDS}s to "
                 f"{PREVIEW_START_SECONDS + PREVIEW_DURATION_SECONDS}s"))
display(Audio(filename=str(preview_original)))
display(Markdown(f"**Model input** — denoise {'on' if DENOISE else 'off'}"))
display(Audio(filename=str(preview_model)))


## 3. NeMo diarization

In [ ]:
import gc
import json
import os
import subprocess
import sys
import time
device, gpu_info = gpu_preflight()
print(gpu_info)
import torch
from nemo.collections.asr.models import SortformerEncLabelModel

MODEL_ID = "nvidia/Nemotron-3-Diarization"
diar_model = SortformerEncLabelModel.from_pretrained(MODEL_ID).to(device).eval()
diar_model.sortformer_modules.spkcache_len = 264
diar_model.sortformer_modules.fifo_len = 40
diar_model.sortformer_modules.chunk_len = 340
diar_model.sortformer_modules.chunk_right_context = 40
diar_model.sortformer_modules.spkcache_update_period = 300
diar_model._check_streaming_parameters()
started = time.perf_counter()
predicted = diar_model.diarize(audio=[str(model_wav)], batch_size=1)[0]
diar_seconds = time.perf_counter() - started
diar_rows = []
for item in predicted:
    start, end, speaker = item.split()
    diar_rows.append(dict(start_seconds=float(start), end_seconds=float(end), speaker=speaker))
import csv
with (run_dir / "diarization.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["start_seconds", "end_seconds", "speaker"])
    writer.writeheader()
    writer.writerows(diar_rows)
with (run_dir / "diarization.rttm").open("w", encoding="utf-8") as handle:
    for row in diar_rows:
        handle.write(f"SPEAKER audio 1 {row['start_seconds']:.3f} "
                     f"{row['end_seconds'] - row['start_seconds']:.3f} <NA> <NA> "
                     f"{row['speaker']} <NA> <NA>\n")
del diar_model
gc.collect()
torch.cuda.empty_cache()
print(f"Saved {len(diar_rows)} diarization intervals in {diar_seconds:.1f}s")


## 4. Faster Whisper in a clean process

In [ ]:
# Run CTranslate2 in a child process so CUDA libraries are visible before import.
ASR_MODEL = "large-v3"
ASR_LANGUAGE = "en"  # use None for automatic language detection
config = dict(wav_path=str(model_wav), output_json=str(run_dir / "asr_raw.json"),
              model=ASR_MODEL, language=ASR_LANGUAGE)
config_path = run_dir / "asr_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
env = os.environ.copy()
if sys.platform == "linux":
    try:
        import nvidia.cublas
        import nvidia.cudnn
        from pathlib import Path
        lib_dirs = [Path(nvidia.cublas.__path__[0]) / "lib",
                    Path(nvidia.cudnn.__path__[0]) / "lib"]
        env["LD_LIBRARY_PATH"] = ":".join(map(str, lib_dirs)) + ":" + env.get("LD_LIBRARY_PATH", "")
    except ImportError:
        print("Using system cuBLAS/cuDNN; install the NVIDIA pip wheels if CTranslate2 cannot find them")
started = time.perf_counter()
subprocess.run([sys.executable, str(Path("run_faster_whisper.py").resolve()), str(config_path)],
               env=env, check=True)
asr_seconds = time.perf_counter() - started
records = json.loads((run_dir / "asr_raw.json").read_text(encoding="utf-8"))
print(f"Saved {len(records)} ASR segments in {asr_seconds:.1f}s")


## 5. Time align words and save results

In [ ]:
# Assign each ASR word by time overlap. Ambiguous/unknown labels stay explicit.
from pipeline_common import save_turns
word_rows = []
for segment_index, record in enumerate(records):
    for word in record.get("words", []):
        start, end = word.get("start"), word.get("end")
        if start is None or end is None or end <= start:
            continue
        overlap = {}
        for interval in diar_rows:
            amount = max(0.0, min(end, interval["end_seconds"]) - max(start, interval["start_seconds"]))
            if amount:
                name = interval["speaker"]
                overlap[name] = overlap.get(name, 0.0) + amount
        ranked = sorted(overlap.items(), key=lambda pair: pair[1], reverse=True)
        length = end - start
        if not ranked or ranked[0][1] < 0.5 * length:
            speaker, reason = "UNKNOWN", "no_clear_time_match"
        elif len(ranked) > 1 and ranked[1][1] >= 0.25 * length:
            speaker, reason = "AMBIGUOUS", "multiple_active_speakers"
        else:
            speaker, reason = ranked[0][0], ""
        word_rows.append(dict(asr_segment=segment_index, start_seconds=start,
                              end_seconds=end, word=word["word"],
                              model_speaker=speaker, review_reason=reason))
with (run_dir / "speaker_attributed_words.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["asr_segment", "start_seconds", "end_seconds",
                                                "word", "model_speaker", "review_reason"])
    writer.writeheader()
    writer.writerows(word_rows)
turns = []
for word in word_rows:
    if (turns and turns[-1]["asr_segment"] == word["asr_segment"]
            and turns[-1]["model_speaker"] == word["model_speaker"]
            and turns[-1]["review_reason"] == word["review_reason"]):
        turns[-1]["end_seconds"] = word["end_seconds"]
        turns[-1]["text"] += word["word"]
    else:
        turns.append(dict(asr_segment=word["asr_segment"], start_seconds=word["start_seconds"],
                          end_seconds=word["end_seconds"], model_speaker=word["model_speaker"],
                          text=word["word"], review_reason=word["review_reason"]))
rows = save_turns(run_dir, turns, duration_seconds)
save_summary(run_dir, diarization_model=MODEL_ID, asr_model=ASR_MODEL, **gpu_info,
             denoise=DENOISE, diarization_seconds=round(diar_seconds, 2),
             asr_seconds=round(asr_seconds, 2), diarization_intervals=len(diar_rows),
             asr_segments=len(records), attributed_turns=len(rows),
             needs_review=not rows or any(row["review_reason"] for row in rows))
print(f"{len(rows)} turns saved in {run_dir}; review flags are in the CSV")
rows[:10]


## 6. Three-panel waveform, diarization, and transcript timeline

In [ ]:
from pipeline_common import render_speaker_timeline, show_figure
fig = render_speaker_timeline(
    run_dir, rows, route="nemo_fw", source_name=Path(INPUT_FILE).name,
    diarization=diar_rows,
)
show_figure(fig)
